## Imports + project root

In [1]:
# %% [markdown]
# # Tournament Simulation
# Simulate FIFA 2026 matches using trained models.

# %%
import os
from pathlib import Path
import sys
import pandas as pd
import joblib

# Set project root
os.chdir(r"c:\Github\fifa2026-prediction")
PROJECT_ROOT = Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("Project root:", PROJECT_ROOT)


Project root: c:\Github\fifa2026-prediction


## Load models and (optionally) Elo data 

In [2]:
# %% [markdown]
# ## Load Models and Data

# %%
models_dir = PROJECT_ROOT / "models"

log_reg = joblib.load(models_dir / "logistic_regression.pkl")
scaler = joblib.load(models_dir / "scaler.pkl")
xgb = joblib.load(models_dir / "xgboost.pkl")

print("✅ Models loaded.")

# Optional: load Elo ratings to look up team strengths
elo_path = PROJECT_ROOT / "data" / "raw" / "elo_ratings.csv"
elo_df = pd.read_csv(elo_path)
print("✅ Elo ratings loaded:", elo_df.shape)
print(elo_df.head())


✅ Models loaded.
✅ Elo ratings loaded: (10, 2)
       team  elo_rating
0    Brazil        1840
1   Germany        1780
2    France        1760
3     Spain        1715
4  Portugal        1710


## Helper: get Elo for a team

In [3]:
# %% [markdown]
# ## Helper: Get Elo Rating for a Team

# %%
def get_elo(team_name, default_elo=1600):
    row = elo_df[elo_df["team"].str.lower() == team_name.lower()]
    if not row.empty:
        return float(row["elo_rating"].iloc[0])
    return default_elo


## Reuse match feature + prediction logic

In [4]:
# %% [markdown]
# ## Feature Builder and Prediction

# %%
def build_match_features(home_team, away_team, home_elo, away_elo,
                         is_home_host=0, is_away_host=0,
                         is_neutral=0, is_friendly=0):

    df = pd.DataFrame([{
        "home_elo": home_elo,
        "away_elo": away_elo,
        "elo_diff": home_elo - away_elo,
        "is_home_host": is_home_host,
        "is_away_host": is_away_host,
        "is_neutral": is_neutral,
        "is_friendly": is_friendly,
    }])

    return df


def predict_match(home_team, away_team,
                  is_home_host=0, is_away_host=0,
                  is_neutral=0, is_friendly=0):

    home_elo = get_elo(home_team)
    away_elo = get_elo(away_team)

    features = build_match_features(
        home_team, away_team, home_elo, away_elo,
        is_home_host, is_away_host, is_neutral, is_friendly
    )

    features_scaled = scaler.transform(features)

    lr_pred = log_reg.predict(features_scaled)[0]
    xgb_pred = xgb.predict(features)[0]

    return {
        "home_team": home_team,
        "away_team": away_team,
        "home_elo": home_elo,
        "away_elo": away_elo,
        "logistic_regression_prediction": "Home Win" if lr_pred == 1 else "Not Home Win",
        "xgboost_prediction": "Home Win" if xgb_pred == 1 else "Not Home Win",
    }


## Define a simple “group” and simulate all matches

In [5]:
# %% [markdown]
# ## Example Group Simulation

# %%
group_a = ["USA", "Mexico", "Brazil", "Germany"]

def simulate_group_round_robin(teams):
    results = []
    for i in range(len(teams)):
        for j in range(i + 1, len(teams)):
            home = teams[i]
            away = teams[j]
            res = predict_match(home, away, is_neutral=1)  # neutral ground
            results.append(res)
    return pd.DataFrame(results)

group_results = simulate_group_round_robin(group_a)
group_results


,home_team,away_team,home_elo,away_elo,logistic_regression_prediction,xgboost_prediction
0,USA,Mexico,1600.0,1600.0,Not Home Win,Not Home Win
1,USA,Brazil,1600.0,1840.0,Not Home Win,Not Home Win
2,USA,Germany,1600.0,1780.0,Not Home Win,Not Home Win
3,Mexico,Brazil,1600.0,1840.0,Not Home Win,Not Home Win
4,Mexico,Germany,1600.0,1780.0,Not Home Win,Not Home Win
5,Brazil,Germany,1840.0,1780.0,Home Win,Home Win


## Turn predictions into “points” and rank the group
We’ll use XGBoost’s prediction as the deciding model and keep it simple:

“Home Win” → home gets 3 points

“Not Home Win” → away gets 3 points (no draws modeled yet)

In [6]:
# %% [markdown]
# ## Compute Group Standings

# %%
def compute_standings_from_results(results_df):
    points = {}

    for _, row in results_df.iterrows():
        home = row["home_team"]
        away = row["away_team"]
        pred = row["xgboost_prediction"]

        points.setdefault(home, 0)
        points.setdefault(away, 0)

        if pred == "Home Win":
            points[home] += 3
        else:
            points[away] += 3

    standings = (
        pd.DataFrame([
            {"team": team, "points": pts}
            for team, pts in points.items()
        ])
        .sort_values("points", ascending=False)
        .reset_index(drop=True)
    )

    return standings

standings_a = compute_standings_from_results(group_results)
standings_a


,team,points
0,Brazil,9
1,Germany,6
2,Mexico,3
3,USA,0


## (Optional) Wrap it into a reusable function

In [7]:
# %% [markdown]
# ## Full Group Simulation Helper

# %%
def simulate_group(teams, group_name="Group"):
    print(f"=== {group_name} ===")
    results = simulate_group_round_robin(teams)
    standings = compute_standings_from_results(results)
    return results, standings

group_a_results, group_a_standings = simulate_group(group_a, "Group A")
group_a_standings


=== Group A ===


,team,points
0,Brazil,9
1,Germany,6
2,Mexico,3
3,USA,0
